# HP Group Analysis

Creatures with identical actual HP in the same CR tier, but different predicted HP.
The prediction spread reveals which features drive the differences — and which costs may need tuning.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# Resolve data directory relative to this notebook's location
NOTEBOOK_DIR = Path.cwd()

# Walk up until we find the data/ directory
for _candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR / '..', NOTEBOOK_DIR / '../..']:
    if (_candidate / 'data' / 'engineered_features.csv').exists():
        DATA_DIR = str(_candidate / 'data')
        break
else:
    DATA_DIR = './data'

df = pd.read_csv(DATA_DIR + '/engineered_features.csv')
contributions_df = pd.read_csv(DATA_DIR + '/feature_contributions.csv')

print(f'Loaded {len(df)} creatures, {len(contributions_df)} contribution rows')

Loaded 324 creatures, 324 contribution rows


In [3]:
# For cr3 (CR 5-10), group creatures within 2 HP of each other.
# For other tiers, use exact HP match.
HP_PROXIMITY = 2
MIN_GROUP_SIZE = 5

def find_hp_groups(df, tier, proximity=0):
    """Find groups of creatures with similar HP in a tier."""
    tier_df = df[df['cr_tier'] == tier].sort_values('actual_hp').copy()
    
    if proximity == 0:
        # Exact match
        tier_df['hp_group'] = tier_df['actual_hp']
    else:
        # Cluster creatures within `proximity` HP of each other
        groups = []
        group_id = 0
        group_min_hp = None
        for _, row in tier_df.iterrows():
            hp = row['actual_hp']
            if group_min_hp is None or hp - group_min_hp > proximity:
                group_id += 1
                group_min_hp = hp
            groups.append(group_id)
        tier_df['hp_group'] = groups
    
    return tier_df

# Build groups for all tiers
all_groups = []
for tier in ['cr1', 'cr2', 'cr3', 'cr4', 'cr5']:
    proximity = HP_PROXIMITY if tier == 'cr3' else 0
    tier_df = find_hp_groups(df, tier, proximity)
    
    for gid, gdf in tier_df.groupby('hp_group'):
        if len(gdf) >= MIN_GROUP_SIZE:
            hp_min, hp_max = int(gdf['actual_hp'].min()), int(gdf['actual_hp'].max())
            hp_label = str(hp_min) if hp_min == hp_max else f'{hp_min}-{hp_max}'
            all_groups.append({
                'cr_tier': tier,
                'hp_label': hp_label,
                'hp_min': hp_min,
                'hp_max': hp_max,
                'count': len(gdf),
                'names': gdf['Name'].tolist(),
                'pred_range': gdf['predicted_hp'].max() - gdf['predicted_hp'].min(),
            })

groups = pd.DataFrame(all_groups).sort_values('pred_range', ascending=False)
print(f'{len(groups)} groups found\n')
groups[['cr_tier', 'hp_label', 'count', 'pred_range']].round(1)

20 groups found



,cr_tier,hp_label,count,pred_range
19,cr3,136-138,7,68.2
14,cr2,45,10,64.7
18,cr3,126-127,7,57.4
11,cr2,22,6,46.3
17,cr3,110-112,6,39.8
15,cr2,52,10,38.6
13,cr2,33,5,31.7
12,cr2,27,5,28.0
8,cr1,13,11,24.6
7,cr1,11,12,22.6


In [4]:
# Phase 2 columns to always show
phase2_cols = [
    'ac_deviation', 'attack_deviation', 'dpr_deviation', 'save_dc_deviation',
    'has_advantage_condition', 'has_disadvantage_condition',
    'has_attackers_advantage', 'inflicts_prone'
]

# Resistance/immunity columns
resist_immun_cols = ['resistance_count', 'immunity_count']

# Feature flag columns (feature_*) and contribution columns (contrib_*)
feature_flag_cols = [c for c in df.columns if c.startswith('feature_') and c not in ['feature_hp', 'feature_dpr', 'feature_ac', 'feature_attack']]
contrib_cols = [c for c in contributions_df.columns if c.startswith('contrib_')]

def build_group_df(names):
    """Build a comparison dataframe for a list of creature names."""
    group = df[df['Name'].isin(names)].sort_values('predicted_hp')
    names_sorted = group['Name'].tolist()
    
    # Start with core prediction columns
    core = ['Name', 'actual_hp', 'predicted_hp', 'hp_delta']
    result = group[core].copy()
    
    # Add phase 2 columns that vary
    for col in phase2_cols:
        if col in group.columns and group[col].nunique() > 1:
            result[col] = group[col].values
    
    # Add resistance/immunity counts that vary
    for col in resist_immun_cols:
        if col in group.columns and group[col].nunique() > 1:
            result[col] = group[col].values
    
    # Add feature flags that vary within the group
    for col in feature_flag_cols:
        if col in group.columns and group[col].nunique() > 1:
            result[col.replace('feature_', 'f_')] = group[col].values
    
    # Add non-zero contribution columns that vary
    group_contribs = contributions_df[contributions_df['Name'].isin(names_sorted)]
    if len(group_contribs) > 0:
        group_contribs = group_contribs.set_index('Name').loc[names_sorted]
        for col in contrib_cols:
            if col in group_contribs.columns:
                vals = group_contribs[col]
                if vals.abs().max() > 0.5 and vals.nunique() > 1:
                    result[col] = vals.values
    
    result = result.set_index('Name')
    return result.round(1)

print(f'Ready — {len(feature_flag_cols)} feature flags, {len(contrib_cols)} contribution cols')

Ready — 91 feature flags, 0 contribution cols


In [5]:
# Generate one dataframe per group, sorted by prediction range (most spread first)
for _, row in groups.iterrows():
    tier, hp_label, count = row['cr_tier'], row['hp_label'], int(row['count'])
    print(f'\n{"="*80}')
    print(f'{tier} | actual_hp={hp_label} | {count} creatures | pred range: {row["pred_range"]:.1f} HP')
    print(f'{"="*80}')
    display(build_group_df(row['names']))


cr3 | actual_hp=136-138 | 7 creatures | pred range: 68.2 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation,attack_deviation,dpr_deviation,save_dc_deviation,resistance_count,immunity_count,f_amphibious,f_angelic_weapons,f_breath_weapon,f_damage_absorption,f_false_appearance,f_innate_spellcasting,f_magic_resistance,f_shapechange,f_siege_monster,f_stench
Name,,,,,,,,,,,,,,,,,,,
Shambling Mound,136,118.9,-17.1,0.0,0.0,-9.0,-1.0,2,1,0,0,0,1,0,0,0,0,0,0
Hezrou,136,136.7,0.7,6.0,0.0,-16.0,-1.0,5,1,0,0,0,0,0,0,1,0,0,1
Young Green Dragon,136,139.3,3.3,7.0,0.0,4.3,-1.0,0,1,1,0,1,0,0,0,0,0,0,0
Tyrannosaurus Rex,136,149.4,13.4,-2.0,3.0,0.5,2.0,0,0,0,0,0,0,0,0,0,0,0,0
Frost Giant,138,159.4,21.4,2.0,2.0,-2.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0
Deva,136,172.7,36.7,4.0,-1.0,-14.0,0.0,3,0,0,1,0,0,0,1,1,1,0,0
Treant,138,187.1,49.1,0.0,2.0,-26.0,0.0,2,0,0,0,0,0,1,0,0,0,1,0



cr2 | actual_hp=45 | 10 creatures | pred range: 64.7 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation,attack_deviation,dpr_deviation,save_dc_deviation,has_disadvantage_condition,inflicts_prone,resistance_count,...,f_etherealness,f_horrifying_visage,f_incorporeal_movement,f_keen_senses,f_life_drain,f_pack_tactics,f_possession,f_rampage,f_spider_climb,f_sunlight_sensitivity
Name,,,,,,,,,,,,,,,,,,,,,
Ghost,45,10.9,-34.1,-3.0,-1.0,-11.0,-1.0,0,0,6,...,1,1,1,0,0,0,1,0,0,0
Silver Dragon Wyrmling,45,12.3,-32.7,8.0,1.0,1.3,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Centaur,45,38.6,-6.4,-1.0,1.0,16.5,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Rhinoceros,45,46.4,1.4,-2.0,2.0,-3.0,2.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
Hunter Shark,45,47.1,2.1,-1.0,5.0,-4.0,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Giant Hyena,45,51.1,6.1,0.0,0.0,0.0,0.0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
Merrow,45,52.1,7.1,0.0,1.0,0.5,0.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Hell Hound,45,65.8,20.8,2.0,1.0,0.7,-1.0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
Wight,45,67.4,22.4,1.0,-1.0,3.0,0.0,1,0,3,...,0,0,0,0,1,0,0,0,0,1



cr3 | actual_hp=126-127 | 7 creatures | pred range: 57.4 HP


...

,actual_hp,predicted_hp,hp_delta,ac_deviation,attack_deviation,dpr_deviation,save_dc_deviation,f_amphibious,f_invisibility,f_keen_senses
Name,,,,,,,,,,
Stirge,2,1.8,-0.2,3.0,2.0,2.5,0.0,0,0,0
Cat,2,4.9,2.9,2.0,-2.0,-1.0,0.0,0,0,1
Poisonous Snake,2,6.4,4.4,2.0,2.0,3.0,-1.0,0,0,0
Crab,2,6.4,4.4,1.0,-2.0,-1.0,0.0,1,0,0
Lizard,2,7.9,5.9,0.0,-2.0,-1.0,0.0,0,0,0
Sprite,2,8.2,6.2,7.0,3.0,-4.0,-1.0,0,1,0



cr1 | actual_hp=3 | 5 creatures | pred range: 5.7 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation,attack_deviation,dpr_deviation,f_hold_breath,f_keen_senses,f_pack_tactics,f_terrain_camouflage
Name,,,,,,,,,,
Eagle,3,-1.8,-4.8,2.0,2.0,2.5,0,1,0,0
Octopus,3,-1.1,-4.1,2.0,2.0,-1.0,1,0,0,1
Baboon,3,-0.3,-3.3,2.0,0.0,1.5,0,0,1,0
Jackal,3,-0.3,-3.3,2.0,0.0,1.5,0,1,1,0
Badger,3,3.9,0.9,0.0,0.0,-1.0,0,1,0,0



cr1 | actual_hp=4 | 5 creatures | pred range: 4.7 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation,attack_deviation,dpr_deviation,inflicts_prone,f_charge,f_illumination,f_sure_footed
Name,,,,,,,,,,
Giant Fire Beetle,4,-0.5,-4.5,3.0,-1.0,2.5,0,0,1,0
Deer,4,-0.1,-4.1,3.0,0.0,0.5,0,0,0,0
Giant Centipede,4,0.4,-3.6,2.0,1.0,10.0,0,0,0,0
Goat,4,0.6,-3.4,0.0,1.0,1.5,1,1,0,1
Commoner,4,4.1,0.1,0.0,0.0,0.5,0,0,0,0


In [6]:
import sys
sys.path.insert(0, '../..')
from notebooks.helper_files.feature_config import (
    PHASE2_PENALTIES, DMG_AC_ADJUSTMENTS, DMG_ATTACK_ADJUSTMENTS,
    DMG_DPR_ADJUSTMENTS, DMG_HP_PER_USE, DMG_HP_BY_TIER,
    DMG_HP_PERCENTAGE, DMG_HP_MULTIPLIER,
    RESISTANCE_PENALTY_PER_COUNT, IMMUNITY_PENALTY_PER_COUNT,
)

def feature_hp_cost(feat_name, tier):
    """Compute the HP cost of a single DMG feature for a given tier."""
    penalties = PHASE2_PENALTIES[tier]
    cost = 0.0
    ac_key = feat_name.replace('_', ' ')
    if ac_key in DMG_AC_ADJUSTMENTS:
        cost += DMG_AC_ADJUSTMENTS[ac_key] * penalties['ac_deviation']
    if ac_key in DMG_ATTACK_ADJUSTMENTS:
        cost += DMG_ATTACK_ADJUSTMENTS[ac_key] * penalties['attack_deviation']
    if feat_name in DMG_DPR_ADJUSTMENTS:
        cost += DMG_DPR_ADJUSTMENTS[feat_name] * penalties['dpr_deviation']
    if feat_name in DMG_HP_PER_USE:
        cost -= DMG_HP_PER_USE[feat_name].get(tier, 0) * 3
    if feat_name in DMG_HP_BY_TIER:
        cost -= DMG_HP_BY_TIER[feat_name].get(tier, 0)
    if feat_name in DMG_HP_PERCENTAGE:
        tier_df = df[df['cr_tier'] == tier]
        mean_baseline = tier_df['hp_baseline'].mean()
        pct = DMG_HP_PERCENTAGE[feat_name]
        cost -= mean_baseline * pct / (1 + pct)
    if feat_name in DMG_HP_MULTIPLIER:
        tier_df = df[df['cr_tier'] == tier]
        mean_baseline = tier_df['hp_baseline'].mean()
        mult = DMG_HP_MULTIPLIER[feat_name]
        cost -= mean_baseline * (1 - 1 / mult)
    return cost

def build_group_cost_df(names):
    """Show HP impact of each feature value. Every cell = value × penalty rate."""
    group = df[df['Name'].isin(names)].sort_values('predicted_hp')
    tier = group['cr_tier'].iloc[0]
    penalties = PHASE2_PENALTIES[tier]

    core = ['Name', 'actual_hp', 'predicted_hp', 'hp_delta']
    result = group[core].copy()

    # Phase 2 columns → value × penalty rate
    for col in phase2_cols:
        if col in group.columns and group[col].nunique() > 1:
            rate = penalties.get(col, 0)
            col_name = f'{col} [×{rate:+.2f}]'
            result[col_name] = (group[col].values * rate).round(1)

    # Resistance/immunity → actual HP penalty per creature
    resist_penalty = (group['hp_after_phase2'] - group['hp_after_resist_immun_penalty']).values
    resist_varies = len(set(resist_penalty.round(1))) > 1
    if resist_varies:
        result['resist+immun penalty'] = (-resist_penalty).round(1)

    # Feature flags → 0/1 × feature HP cost
    for col in feature_flag_cols:
        if col in group.columns and group[col].nunique() > 1:
            feat_name = col.replace('feature_', '')
            cost = feature_hp_cost(feat_name, tier)
            cost_label = f'{cost:+.1f}' if cost != 0 else 'no cost'
            col_name = f'f_{feat_name} [{cost_label}]'
            result[col_name] = (group[col].values * cost).round(1) if cost != 0 else group[col].values

    result = result.set_index('Name')
    return result.round(1)

print('Ready')

Ready


In [7]:
# Same groups, but feature columns show HP cost instead of 0/1 flags.
# Column headers include the per-creature cost in brackets.
for _, row in groups.iterrows():
    tier, hp_label, count = row['cr_tier'], row['hp_label'], int(row['count'])
    print(f'\n{"="*80}')
    print(f'{tier} | actual_hp={hp_label} | {count} creatures | pred range: {row["pred_range"]:.1f} HP')
    print(f'{"="*80}')
    display(build_group_cost_df(row['names']))


cr3 | actual_hp=136-138 | 7 creatures | pred range: 68.2 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation [×-3.50],attack_deviation [×-4.00],dpr_deviation [×-1.25],save_dc_deviation [×-6.00],resist+immun penalty,f_amphibious [no cost],f_angelic_weapons [no cost],f_breath_weapon [no cost],f_damage_absorption [no cost],f_false_appearance [no cost],f_innate_spellcasting [no cost],f_magic_resistance [-7.0],f_shapechange [no cost],f_siege_monster [no cost],f_stench [-3.5]
Name,,,,,,,,,,,,,,,,,,
Shambling Mound,136,118.9,-17.1,-0.0,-0.0,11.2,6.0,-0.0,0,0,0,1,0,0,-0.0,0,0,-0.0
Hezrou,136,136.7,0.7,-21.0,-0.0,20.0,6.0,-10.5,0,0,0,0,0,0,-7.0,0,0,-3.5
Young Green Dragon,136,139.3,3.3,-24.5,-0.0,-5.4,6.0,-0.0,1,0,1,0,0,0,-0.0,0,0,-0.0
Tyrannosaurus Rex,136,149.4,13.4,7.0,-12.0,-0.6,-12.0,-0.0,0,0,0,0,0,0,-0.0,0,0,-0.0
Frost Giant,138,159.4,21.4,-7.0,-8.0,2.5,-0.0,-0.0,0,0,0,0,0,0,-0.0,0,0,-0.0
Deva,136,172.7,36.7,-14.0,4.0,17.5,-0.0,-12.1,0,1,0,0,0,1,-7.0,1,0,-0.0
Treant,138,187.1,49.1,-0.0,-8.0,32.5,-0.0,-0.0,0,0,0,0,1,0,-0.0,0,1,-0.0



cr2 | actual_hp=45 | 10 creatures | pred range: 64.7 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation [×-2.50],attack_deviation [×-3.00],dpr_deviation [×-1.00],save_dc_deviation [×-5.00],has_disadvantage_condition [×+1.50],inflicts_prone [×-1.12],resist+immun penalty,...,f_etherealness [no cost],f_horrifying_visage [-12.8],f_incorporeal_movement [no cost],f_keen_senses [no cost],f_life_drain [no cost],f_pack_tactics [-3.0],f_possession [-31.9],f_rampage [-2.0],f_spider_climb [no cost],f_sunlight_sensitivity [no cost]
Name,,,,,,,,,,,,,,,,,,,,,
Ghost,45,10.9,-34.1,7.5,3.0,11.0,5.0,0.0,-0.0,-21.3,...,1,-12.8,1,0,0,-0.0,-31.9,-0.0,0,0
Silver Dragon Wyrmling,45,12.3,-32.7,-20.0,-3.0,-1.3,-0.0,0.0,-0.0,-0.0,...,0,-0.0,0,0,0,-0.0,-0.0,-0.0,0,0
Centaur,45,38.6,-6.4,2.5,-3.0,-16.5,-0.0,0.0,-0.0,-0.0,...,0,-0.0,0,0,0,-0.0,-0.0,-0.0,0,0
Rhinoceros,45,46.4,1.4,5.0,-6.0,3.0,-10.0,0.0,-1.1,-0.0,...,0,-0.0,0,0,0,-0.0,-0.0,-0.0,0,0
Hunter Shark,45,47.1,2.1,2.5,-15.0,4.0,-0.0,0.0,-0.0,-0.0,...,0,-0.0,0,0,0,-0.0,-0.0,-0.0,0,0
Giant Hyena,45,51.1,6.1,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,...,0,-0.0,0,0,0,-0.0,-0.0,-2.0,0,0
Merrow,45,52.1,7.1,-0.0,-3.0,-0.5,-0.0,0.0,-0.0,-0.0,...,0,-0.0,0,0,0,-0.0,-0.0,-0.0,0,0
Hell Hound,45,65.8,20.8,-5.0,-3.0,-0.7,5.0,0.0,-0.0,-0.0,...,0,-0.0,0,1,0,-3.0,-0.0,-0.0,0,0
Wight,45,67.4,22.4,-2.5,3.0,-3.0,-0.0,1.5,-0.0,-9.6,...,0,-0.0,0,0,1,-0.0,-0.0,-0.0,0,1



cr3 | actual_hp=126-127 | 7 creatures | pred range: 57.4 HP


...

,actual_hp,predicted_hp,hp_delta,ac_deviation [×-1.50],attack_deviation [×-2.00],dpr_deviation [×-0.75],save_dc_deviation [×-3.50],f_amphibious [no cost],f_invisibility [-1.5],f_keen_senses [no cost]
Name,,,,,,,,,,
Stirge,2,1.8,-0.2,-4.5,-4.0,-1.9,-0.0,0,-0.0,0
Cat,2,4.9,2.9,-3.0,4.0,0.8,-0.0,0,-0.0,1
Poisonous Snake,2,6.4,4.4,-3.0,-4.0,-2.2,3.5,0,-0.0,0
Crab,2,6.4,4.4,-1.5,4.0,0.8,-0.0,1,-0.0,0
Lizard,2,7.9,5.9,-0.0,4.0,0.8,-0.0,0,-0.0,0
Sprite,2,8.2,6.2,-10.5,-6.0,3.0,3.5,0,-1.5,0



cr1 | actual_hp=3 | 5 creatures | pred range: 5.7 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation [×-1.50],attack_deviation [×-2.00],dpr_deviation [×-0.75],resist+immun penalty,f_hold_breath [no cost],f_keen_senses [no cost],f_pack_tactics [-2.0],f_terrain_camouflage [no cost]
Name,,,,,,,,,,,
Eagle,3,-1.8,-4.8,-3.0,-4.0,-1.9,3.3,0,1,-0.0,0
Octopus,3,-1.1,-4.1,-3.0,-4.0,0.8,1.3,1,0,-0.0,1
Baboon,3,-0.3,-3.3,-3.0,-0.0,-1.1,-0.0,0,0,-2.0,0
Jackal,3,-0.3,-3.3,-3.0,-0.0,-1.1,-0.0,0,1,-2.0,0
Badger,3,3.9,0.9,-0.0,-0.0,0.8,-0.0,0,1,-0.0,0



cr1 | actual_hp=4 | 5 creatures | pred range: 4.7 HP


,actual_hp,predicted_hp,hp_delta,ac_deviation [×-1.50],attack_deviation [×-2.00],dpr_deviation [×-0.75],inflicts_prone [×-0.75],resist+immun penalty,f_charge [no cost],f_illumination [no cost],f_sure_footed [no cost]
Name,,,,,,,,,,,
Giant Fire Beetle,4,-0.5,-4.5,-4.5,2.0,-1.9,-0.0,-0.0,0,1,0
Deer,4,-0.1,-4.1,-4.5,-0.0,-0.4,-0.0,0.3,0,0,0
Giant Centipede,4,0.4,-3.6,-3.0,-2.0,-7.5,-0.0,-0.0,0,0,0
Goat,4,0.6,-3.4,-0.0,-2.0,-1.1,-0.8,-0.0,1,0,1
Commoner,4,4.1,0.1,-0.0,-0.0,-0.4,-0.0,-0.0,0,0,0
